In [2]:
# test to see if the dataset loads
import pandas as pd
import numpy as np
# Load the local CSV file after downloading, the file is in the data/ directory
df = pd.read_csv('../data/stock_details_5_years.csv')

# Display head of the dataset to get basic info
df.head(5)

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company
0,2018-11-29 00:00:00-05:00,43.829761,43.863354,42.639594,43.083508,167080000,0.00,0.0,AAPL
1,2018-11-29 00:00:00-05:00,104.769074,105.519257,103.534595,104.636131,28123200,0.00,0.0,MSFT
2,2018-11-29 00:00:00-05:00,54.176498,55.007500,54.099998,54.729000,31004000,0.00,0.0,GOOGL
3,2018-11-29 00:00:00-05:00,83.749496,84.499496,82.616501,83.678497,132264000,0.00,0.0,AMZN
4,2018-11-29 00:00:00-05:00,39.692784,40.064904,38.735195,39.037853,54917200,0.04,0.0,NVDA


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 602962 entries, 0 to 602961
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Date          602962 non-null  str    
 1   Open          602962 non-null  float64
 2   High          602962 non-null  float64
 3   Low           602962 non-null  float64
 4   Close         602962 non-null  float64
 5   Volume        602962 non-null  int64  
 6   Dividends     602962 non-null  float64
 7   Stock Splits  602962 non-null  float64
 8   Company       602962 non-null  str    
dtypes: float64(6), int64(1), str(2)
memory usage: 41.4 MB


In [4]:
# Vectorized* Boolean Mask / Transformation 
# boolean column indicates a gain, where the stock closed higher than it opened
df["Gain"] = df["Close"] > df["Open"]

df["Gain"].head()

0    False
1    False
2     True
3    False
4    False
Name: Gain, dtype: bool

In [5]:
# Map utc dates to day names

#updated the date column to utc
#then created a column that gets the day name from the utc, and maps
df["Date"] = pd.to_datetime(df["Date"], utc=True)  # normalize to UTC

df["Day_Name"] = df["Date"].dt.dayofweek.map({
    0: "Monday", 1: "Tuesday", 2: "Wednesday",
    3: "Thursday", 4: "Friday",
    5: "Saturday", 6: "Sunday"
})

df["Day_Name"].head()

0    Thursday
1    Thursday
2    Thursday
3    Thursday
4    Thursday
Name: Day_Name, dtype: str

In [6]:
# column with df.apply

# more descriptive than "gain", and not binary, helps us get more information on the stock's daily price instead of just a binary result

def stock_move(row):
    if row["Close"] > row["Open"]:
        return "Gain"
    elif row["Close"] < row["Open"]:
        return "Loss"
    else:
        return "Break Even" #although kind of uncommon

df["Stock_Move"] = df.apply(stock_move, axis="columns")

df["Stock_Move"].head()

0    Loss
1    Loss
2    Gain
3    Loss
4    Loss
Name: Stock_Move, dtype: str

In [8]:
# Missing Data Handling

#fill missing dividents with 0 if needed (dataset currently has no missing values)
df["Dividends"] = df["Dividends"].fillna(0)

df["Dividends"].head()

#categorical bucketing

df["Volume_Bucket"] = pd.cut(
    df["Volume"],
    bins=[0, 1e7, 5e7, 1e8, 5e8, np.inf], #0 - 10mil is cut off for "Low", etc.. np.inf meaning anything above to infinity
    #in scientific notation                >100 mill is very high, 500m+ is extreme
    labels=["Low", "Medium", "High", "Very High", "Extreme"]
)

df["Volume_Bucket"].head()

0    Very High
1       Medium
2       Medium
3    Very High
4         High
Name: Volume_Bucket, dtype: category
Categories (5, str): ['Low' < 'Medium' < 'High' < 'Very High' < 'Extreme']

In [10]:
#value counts

#this helps us interpret what companies are traded the most and the least during this time period
#it also helps us see which companies haev the most historical records
#all stocks trade daily besides holidays or weekends duringg this period, counts should be similar for some companies
# but some comapnies taht ipo'd (term for when a stock can be traded by the public) at a later date than the start date of the dataset will have less records
company_counts = df["Company"].value_counts()

print(company_counts.head())
company_counts.tail()

#grouby(..).agg(..)

# This shows which companies have had the highest average stock price
# over the full dataset window.
avg_close = df.groupby("Company")["Close"].agg("mean").sort_values(ascending=False)
print(avg_close.head())

Company
AAPL     1258
MSFT     1258
GOOGL    1258
AMZN     1258
NVDA     1258
Name: count, dtype: int64
Company
NVR     4369.628335
BKNG    2126.469873
AZO     1645.242242
CMG     1324.551017
MTD     1107.243361
Name: Close, dtype: float64
